# CS172 Computer Vision I: CAPTCHA Recognition

Personal implementation of an individual coursework project.

## Report checklist

- Run data generation, training, and the predefined ablations before exporting the final report.
- Modify any helper function when its assumptions do not match your implementation.
- Execute all cells from a clean kernel and keep the outputs visible.
- Keep test data separate from model selection and hyperparameter tuning.
- Give every figure labeled axes, a title or caption, and a short interpretation.
- Export the completed notebook to PDF before submission.

## 0. Setup

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import ConcatDataset, DataLoader, Dataset
from torchvision.io import ImageReadMode, decode_image
from torchvision.transforms import v2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")

## 1. Data generation

Demonstrate valid five-digit CAPTCHAs with random colors, positions, rotations, interference lines, pixel noise, and a border.

In [ ]:
from cs172.generate_captcha import generate_captcha

generated_samples = [
    (np.asarray(image), label)
    for seed in range(8)
    for image, label in [generate_captcha(reference=True, seed=seed)]
]

In [ ]:
def show_captcha_samples(
    samples: list[tuple[np.ndarray, list[int] | str]], columns=4, title=None
):
    rows = math.ceil(len(samples) / columns)
    fig, axes = plt.subplots(
        rows, columns, figsize=(4 * columns, 2.5 * rows), squeeze=False
    )
    for axis in axes.flat:
        axis.axis("off")
    for axis, (image, label) in zip(axes.flat, samples):
        assert isinstance(image, np.ndarray)
        assert image.dtype == np.uint8 and image.ndim == 3 and image.shape[2] == 3
        axis.imshow(image)
        axis.set_title("label: " + "".join(str(value) for value in label))
        axis.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


show_captcha_samples(generated_samples, title="Generated CAPTCHA samples")

## 2. Dataset and preprocessing

Verify the dataset split, preprocessing, and augmentation. The DataLoader batch below reports image/label shapes and data types, and displays its eight images.

In [ ]:
from cs172.datasets import CAPTCHA

train_augmentation = v2.Compose([
    v2.RandomAffine(degrees=4, translate=(0.03, 0.02)),
    v2.ColorJitter(brightness=0.08, contrast=0.08, saturation=0.08),
])
train_dataset = CAPTCHA("data/training_reference.zip", target_size=(80, 260), transform=train_augmentation)
val_dataset = CAPTCHA("data/validation_reference.zip", target_size=(80, 260))

In [ ]:
def inspect_dataset_sample(dataset: Dataset, name: str):
    image, label = dataset[0]
    print(f"{name} size: {len(dataset):,}")
    print(f"{name}[0] image: shape={tuple(image.shape)}, dtype={image.dtype}")
    label_text = "".join(str(int(value)) for value in label)
    print(
        f"{name}[0] label: shape={tuple(label.shape)}, dtype={label.dtype}, value={label_text}"
    )
    assert isinstance(image, torch.Tensor), "expected a torch.Tensor image"
    assert image.ndim == 3 and image.shape[0] == 3, "expected an RGB [3, H, W] image"
    assert image.dtype in (
        torch.uint8,
        torch.float32,
    ), "expected uint8 or float32 image data"
    assert tuple(label.shape) == (5,), "expected a five-digit label"
    assert label.dtype == torch.long, "expected torch.long labels"


inspect_dataset_sample(train_dataset, "training dataset")
inspect_dataset_sample(val_dataset, "validation dataset")

In [ ]:
# Inspect and visualize a real batch yielded by the training DataLoader.
preview_batch_size = 8
preview_loader = DataLoader(
    train_dataset, batch_size=preview_batch_size, shuffle=False, num_workers=0
)
batch_images, batch_labels = next(iter(preview_loader))
image_min = batch_images.min().item()
image_max = batch_images.max().item()
print(
    f"DataLoader batch images: shape={tuple(batch_images.shape)}, "
    f"dtype={batch_images.dtype}, value_range=[{image_min}, {image_max}]"
)
print(
    f"DataLoader batch labels: shape={tuple(batch_labels.shape)}, "
    f"dtype={batch_labels.dtype} (torch.long)"
)
print(
    "Batch labels:",
    ["".join(str(int(value)) for value in label) for label in batch_labels],
)
assert batch_images.shape == (preview_batch_size, 3, 80, 260)
assert batch_images.dtype in (torch.uint8, torch.float32)
assert batch_labels.shape == (preview_batch_size, 5)
assert batch_labels.dtype == torch.long
if batch_images.dtype == torch.uint8:
    assert image_min >= 0 and image_max <= 255
else:
    assert image_min >= 0.0 and image_max <= 1.0

fig, axes = plt.subplots(2, 4, figsize=(14, 4), squeeze=False)
for axis, image, label in zip(axes.flat, batch_images, batch_labels):
    image = image.detach().cpu()
    if image.dtype == torch.uint8:
        image = v2.ToDtype(torch.float32, scale=True)(image)
    label_text = "".join(str(int(value)) for value in label)
    axis.imshow(image.permute(1, 2, 0))
    axis.set_title(f"label: {label_text}")
    axis.axis("off")
fig.suptitle("Training DataLoader batch (8 images)")
plt.tight_layout()
plt.show()


In [ ]:
def show_dataset_indices(dataset: Dataset, indices: list[int], title: str, columns=4):
    rows = math.ceil(len(indices) / columns)
    fig, axes = plt.subplots(
        rows, columns, figsize=(4 * columns, 2.5 * rows), squeeze=False
    )
    for axis, index in zip(axes.flat, indices):
        image, label = dataset[index]
        assert isinstance(image, torch.Tensor)
        assert image.ndim == 3 and image.shape[0] == 3
        assert image.dtype in (torch.uint8, torch.float32)
        image = image.detach().cpu()
        if image.dtype == torch.uint8:
            image = v2.ToDtype(torch.float32, scale=True)(image)

        assert image.min().item() >= 0.0 and image.max().item() <= 1.0
        label_text = "".join(str(int(value)) for value in label)
        axis.imshow(image.permute(1, 2, 0))
        axis.set_title(f"label: {label_text}")
        axis.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


# Repeating index 0 should reveal random training augmentation, if enabled.
show_dataset_indices(
    train_dataset, [0] * 8, "Repeated draws of one augmented training sample"
)

# Distinct validation samples should use deterministic preprocessing only.
val_indices = list(range(min(8, len(val_dataset)))) if val_dataset is not None else []
show_dataset_indices(
    val_dataset, val_indices, "Validation samples without random augmentation"
)

## 3. Model

Verify the interface `[batch, 3, height, width] -> [batch, 10 classes, 5 digits]`, and report parameters as well as computational cost. The profiler below estimates FLOPs only for operators supported by PyTorch's profiler, so state the input size and treat the result as an approximation.

In [ ]:
from cs172.models import CAPTCHAResNet
model = CAPTCHAResNet(image_size=(80, 260))

MODEL_CHECK_BATCH_SIZE = 4
MODEL_INPUT_SIZE = (80, 260)
MODEL_INPUT_DTYPE = torch.uint8

In [ ]:
height, width = MODEL_INPUT_SIZE
if MODEL_INPUT_DTYPE == torch.uint8:
    random_images = torch.randint(0, 256, (MODEL_CHECK_BATCH_SIZE, 3, height, width), dtype=torch.uint8)
elif MODEL_INPUT_DTYPE == torch.float32:
    random_images = torch.rand(MODEL_CHECK_BATCH_SIZE, 3, height, width, dtype=torch.float32)
else:
    raise ValueError(f"Unsupported model input dtype: {MODEL_INPUT_DTYPE}")
random_labels = torch.randint(0, 10, (MODEL_CHECK_BATCH_SIZE, 5), dtype=torch.long)

print(f"random images: {tuple(random_images.shape)}, {random_images.dtype}")
print(f"random labels: {tuple(random_labels.shape)}, {random_labels.dtype}")

model_device = next(model.parameters()).device
model.eval()
with torch.inference_mode():
    random_logits = model(random_images.to(model_device))
print(f"model output: {tuple(random_logits.shape)}, {random_logits.dtype}")
assert tuple(random_logits.shape) == (MODEL_CHECK_BATCH_SIZE, 10, 5)

In [ ]:
from cs172.models import create_model

def compute_num_params_and_flops(model, example_images):
    total_parameters = sum(parameter.numel() for parameter in model.parameters())
    trainable_parameters = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    model_device = next(model.parameters()).device
    activities = [torch.profiler.ProfilerActivity.CPU]
    if model_device.type == "cuda":
        activities.append(torch.profiler.ProfilerActivity.CUDA)
    model.eval()
    with torch.inference_mode(), torch.profiler.profile(activities=activities, with_flops=True, acc_events=True) as profile:
        model(example_images[:1].to(model_device))
    estimated_flops = sum(event.flops or 0 for event in profile.key_averages())
    return {
        "total_parameters": total_parameters,
        "trainable_parameters": trainable_parameters,
        "estimated_flops_per_image": estimated_flops,
    }

cost_models = {
    "Selected CAPTCHAResNet (spatial head)": model,
    "TorchVision ResNet-18 (global average pooling)": create_model("resnet18", image_size=MODEL_INPUT_SIZE),
    "TorchVision ResNet-34 (global average pooling)": create_model("resnet34", image_size=MODEL_INPUT_SIZE),
    "TorchVision ResNet-18 (spatial flatten)": create_model("resnet18_spatial", image_size=MODEL_INPUT_SIZE),
}
for model_name, candidate in cost_models.items():
    cost = compute_num_params_and_flops(candidate, random_images)
    print(
        f"{model_name}: {cost['total_parameters']:,} parameters; "
        f"estimated {cost['estimated_flops_per_image']:,} FLOPs/image"
    )


## 4. Training report

Select the best checkpoint using validation full-image accuracy, never test performance. Plot training and validation loss, per-digit accuracy, and full-image accuracy over epochs.

### Final training configuration

| Setting | Value |
|---|---|
| Seed | 42 |
| Model | Spatial CAPTCHAResNet (ResNet-18 blocks, no global average pooling) |
| Training images | 100,000 `--reference` generated images (seeds 42–100,041, JPEG quality 60–95) |
| Validation images | 10,000 independently generated `--reference` images (seeds 100,042–110,041) |
| Optimizer | AdamW |
| Batch size | 64 |
| Initial learning rate and schedule | 0.001, cosine annealing |
| Weight decay | 0.0001 |
| Data augmentation | RandomAffine and ColorJitter, training only |
| Epochs and best-checkpoint epoch | 10; epoch 10 selected by validation full-image accuracy |

The reference generator renders digits with Pillow's default font at 30–40 px, uses the measured margins and minimum spacing, and samples the five difficulty levels uniformly. It recombines the factors independently in 20% of images, uses another sans font in 10%, and uses looser margins and spacing in 15%, so the model does not rely on exact pixels. The first training attempt stopped after epoch 2 with a Windows shared-memory error (code 1455). Training resumed from the epoch-2 checkpoint, including optimizer and scheduler state, with one data-loading worker.

### TensorBoard curve reader

The functions below read scalar summaries directly from a TensorBoard log directory. The tag mapping must match the names recorded by your training pipeline. TensorBoard is optional and can be installed with `pip install tensorboard`. 

Leveraging Tensorboard to plot curves is just an example. **You can save the metrics in any format you prefer, as long as you can plot them.**

In [ ]:
import json

def read_tensorboard_scalars(log_dir):
    try:
        from tensorboard.backend.event_processing.event_accumulator import (
            EventAccumulator,
        )
    except ImportError as error:
        raise ImportError("Install TensorBoard to read event files.") from error

    accumulator = EventAccumulator(str(log_dir))
    accumulator.Reload()
    history = {}
    for tag in accumulator.Tags().get("scalars", []):
        events = accumulator.Scalars(tag)
        history[tag] = {
            "step": [event.step for event in events],
            "value": [event.value for event in events],
        }
    return history


def plot_training_curves(history, tag_map):
    metrics = ("loss", "digit_acc", "image_acc")
    titles = ("Loss", "Per-digit accuracy", "Full-image accuracy")
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    for axis, metric, title in zip(axes, metrics, titles):
        for split in ("train", "val"):
            tag = tag_map[f"{split}_{metric}"]
            series = history[tag]
            axis.plot(series["step"], series["value"], label=split)
        axis.set(xlabel="epoch", title=title)
        axis.grid(alpha=0.25)
        axis.legend()
    plt.show()

In [ ]:
# The training script writes history.json for direct plotting; TensorBoard is optional.
RUN_CONFIGS = sorted(Path("outputs/reference").glob("*/config.json"))
TENSORBOARD_TAGS = {
    "train_loss": "train/loss",
    "val_loss": "val/loss",
    "train_digit_acc": "train/digit_acc",
    "val_digit_acc": "val/digit_acc",
    "train_image_acc": "train/image_acc",
    "val_image_acc": "val/image_acc",
}

if RUN_CONFIGS:
    history_path = RUN_CONFIGS[-1].parent / "history.json"
    final_history = json.loads(history_path.read_text(encoding="utf-8"))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    for axis, metric, title in zip(axes, ("loss", "digit_acc", "image_acc"), ("Loss", "Per-digit accuracy", "Full-image accuracy")):
        for split in ("train", "val"):
            axis.plot([row["epoch"] for row in final_history], [row[f"{split}_{metric}"] for row in final_history], label=split)
        axis.set(xlabel="epoch", title=title)
        axis.grid(alpha=0.25)
        axis.legend()
    plt.show()
    best_record = max(final_history, key=lambda row: row['val_image_acc'])
    print(f"best epoch: {best_record['epoch']}; train full-image accuracy {best_record['train_image_acc']:.2%}; validation full-image accuracy {best_record['val_image_acc']:.2%}")

### Training analysis

The reference run reached 89.33% validation full-image accuracy after epoch 1 and 98.36% after epoch 2. It dipped to 97.69% at epoch 3 and 98.71% at epoch 6, then rose steadily as the cosine schedule lowered the learning rate: 99.67%, 99.79%, 99.89%, and 99.90% for epochs 7–10. Epoch 10 is selected because it has the highest validation full-image accuracy (99.978% per digit). Training and validation full-image accuracy at epoch 10 were 99.91% and 99.90%, so the run shows no overfitting gap.

The test scores below were calculated after the checkpoint was fixed. The training and validation loaders use separately generated datasets, and checkpoint selection uses validation full-image accuracy. As the assignment directs, the provided CAPTCHA examples informed the synthetic appearance design; these scores describe performance on the required groups rather than an independent estimate on a new CAPTCHA distribution.

## 5. Performance evaluation

After model selection was complete, the epoch-10 checkpoint was evaluated on the five provided test groups (500 images total). Following the assignment instructions, their CAPTCHA examples informed the synthetic appearance design. Training and validation loaders use independently generated datasets; test labels and accuracy are not used for training or checkpoint selection.

In [ ]:
# The checkpoint is fixed by validation full-image accuracy; test results do not change it.
import json
from pathlib import Path
from cs172.models import create_model

run_dir = Path("outputs/reference/20261031-135246-22a31c6b")
run_config = json.loads((run_dir / "config.json").read_text(encoding="utf-8"))
checkpoint_path = run_dir / "checkpoints" / "best.pt"
checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
final_model = create_model(run_config["model"], image_size=tuple(run_config["image_size"])).to(DEVICE)
final_model.load_state_dict(checkpoint["model"])
final_model.eval()
print(f"Selected checkpoint: {checkpoint_path} (epoch {checkpoint['epoch'] + 1})")
print("Selection metric: validation full-image accuracy; test data were not used.")

### Evaluation helpers

This small evaluation-only `Dataset` loads RGB images, parses labels from filenames, and applies deterministic preprocessing. The five datasets are concatenated below so the reported metrics cover all 500 test images.

In [ ]:
# CAUTION: DO NOT modify the code below. It will be used to evaluate your model on the hidden test set.

class EvaluationDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_paths = sorted(
            path for path in Path(image_dir).iterdir()
            if path.suffix.lower() in {".png", ".jpg", ".jpeg"}
        )
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        image_path = self.image_paths[index]
        image = decode_image(str(image_path), mode=ImageReadMode.RGB)
        label_text = image_path.stem.split("#", maxsplit=1)[0]
        label = torch.tensor([int(digit) for digit in label_text])
        if self.transform is not None:
            image = self.transform(image)
        return image, label

@torch.inference_mode()
def evaluate_accuracy(model: torch.nn.Module, loader: DataLoader, device: torch.device):
    model.eval()
    correct_digits = 0
    correct_images = 0
    num_digits = 0
    num_images = 0

    for images, targets in loader:
        images = images.to(device)
        targets = targets.to(device)
        logits = model(images)
        assert logits.shape == (targets.shape[0], 10, 5)
        predictions = logits.argmax(dim=1)
        correct = predictions.eq(targets)
        correct_digits += correct.sum().item()
        correct_images += correct.all(dim=1).sum().item()
        num_digits += targets.numel()
        num_images += targets.shape[0]

    digit_accuracy = correct_digits / num_digits
    image_accuracy = correct_images / num_images
    score = (
        20 * max(digit_accuracy - 0.95, 0.0) / 0.05
        + 20 * max(image_accuracy - 0.90, 0.0) / 0.10
    )
    return {
        "num_images": num_images,
        "digit_acc": digit_accuracy,
        "image_acc": image_accuracy,
        "score": score,
    }

In [ ]:
# Evaluate the fixed checkpoint with the provided evaluate_accuracy helper.
TEST_DIRS = [Path("data/test") / f"test{index}" for index in range(5)]
test_datasets = {path.name: EvaluationDataset(path) for path in TEST_DIRS}
results = {
    name: evaluate_accuracy(final_model, DataLoader(dataset, batch_size=100, shuffle=False), DEVICE)
    for name, dataset in test_datasets.items()
}
aggregate = evaluate_accuracy(
    final_model, DataLoader(ConcatDataset(list(test_datasets.values())), batch_size=100, shuffle=False), DEVICE
)
final_test_report = {"checkpoint": str(checkpoint_path), "results": results, "aggregate": aggregate}
for name, row in results.items():
    print(f"{name}: {row['num_images']} images | per-digit {row['digit_acc']:.2%} | full-image {row['image_acc']:.2%}")
print(f"aggregate: {aggregate['num_images']} images | per-digit {aggregate['digit_acc']:.2%} | full-image {aggregate['image_acc']:.2%} | score {aggregate['score']:.2f}")

In [ ]:
# Predictions next to ground truth: four images per folder, plus every full-image failure.
@torch.inference_mode()
def predict_folder(dataset):
    images = torch.stack([dataset[index][0] for index in range(len(dataset))])
    labels = torch.stack([dataset[index][1] for index in range(len(dataset))])
    predictions = final_model(images.to(DEVICE)).argmax(dim=1).cpu()
    return images, labels, predictions

shown, failures = [], []
for name, dataset in test_datasets.items():
    images, labels, predictions = predict_folder(dataset)
    for index in range(len(dataset)):
        item = (name, images[index], labels[index], predictions[index])
        if not predictions[index].equal(labels[index]):
            failures.append(item)
        if index < 4:
            shown.append(item)

def show_predictions(items, title, columns=4):
    rows = math.ceil(len(items) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(4 * columns, 1.9 * rows), squeeze=False)
    for axis in axes.flat:
        axis.axis("off")
    for axis, (name, image, label, prediction) in zip(axes.flat, items):
        axis.imshow(image.permute(1, 2, 0))
        gt = "".join(map(str, label.tolist()))
        pred = "".join(map(str, prediction.tolist()))
        axis.set_title(f"{name} | GT {gt} | pred {pred}", color="black" if gt == pred else "red", fontsize=10)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show_predictions(shown, "Test predictions (title: ground truth and prediction)")
print(f"full-image failures: {len(failures)} / {sum(len(d) for d in test_datasets.values())}")
if failures:
    show_predictions(failures, "All full-image failures (red titles)")

## 6. Analysis and ablations

For every comparison, change only the named factor and keep unrelated settings fixed. Select each run's checkpoint using validation full-image accuracy. When showing curves over epochs, plot validation metrics. When comparing final performance, report the aggregate test results directly. Use simple plots or tables and state a concise conclusion supported by the results.

The ablation runs below are completed baseline-generator comparisons. The final model above uses the `--reference` generator; its validation curve is shown separately. The ablations were run earlier with the baseline generator, so their absolute test scores are much lower than the final model's; they compare settings while keeping other factors fixed. Every checkpoint was selected using validation full-image accuracy, and the reported test results summarize performance on the assignment's provided groups.

In [ ]:
ABLATION_DIR = Path('outputs/ablations')
ABLATION_STUDIES = ('size', 'augmentation', 'weight-decay', 'batch-size', 'learning-rate', 'backbone', 'pooling')
for study in ABLATION_STUDIES:
    summary_path = ABLATION_DIR / f'{study}_summary.json'
    if not summary_path.is_file():
        print(f'{study}: run `python scripts/run_experiments.py --study {study}` to produce results')
        continue
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    runs = summary['runs']
    labels = [run['name'] for run in runs]
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
    axes = axes.ravel()
    x = np.arange(len(runs))
    axes[0].plot(x, [run['aggregate']['digit_acc'] for run in runs], marker='o', label='per-digit')
    axes[0].plot(x, [run['aggregate']['image_acc'] for run in runs], marker='s', label='full-image')
    axes[0].set(xticks=x, xticklabels=labels, ylabel='test accuracy', title=f'{study}: fixed test comparison')
    axes[0].tick_params(axis='x', rotation=25)
    axes[0].grid(alpha=0.25)
    axes[0].legend()
    metric_specs = (('val_loss', 'validation loss'), ('val_digit_acc', 'validation per-digit accuracy'), ('val_image_acc', 'validation full-image accuracy'))
    for run in runs:
        history_path = Path(run['config']).parent / 'history.json'
        history = json.loads(history_path.read_text(encoding='utf-8'))
        epochs = [row['epoch'] for row in history]
        for axis, (metric, ylabel) in zip(axes[1:], metric_specs):
            axis.plot(epochs, [row[metric] for row in history], label=run['name'])
    for axis, (metric, ylabel) in zip(axes[1:], metric_specs):
        axis.set(xlabel='epoch', ylabel=ylabel, title=f'{study}: {ylabel}')
        axis.grid(alpha=0.25)
        axis.legend(fontsize='small')
    plt.show()
    for run in runs:
        per_set = ', '.join(f"{Path(name).name} {result['image_acc']:.1%}" for name, result in run['test_results'].items())
        print(f"{run['name']}: train time {run['training_seconds'] / 60:.1f} min; test {per_set}; aggregate digit {run['aggregate']['digit_acc']:.2%}, image {run['aggregate']['image_acc']:.2%}")

### 6.1 Training-set size

Train with at least three training-set sizes while keeping the other settings fixed.

**Results.** The ablation cell plots validation curves and aggregate test accuracy for 10k, 20k, and 100k training images. Their best validation full-image accuracies were 99.86% (epoch 10), 99.90% (epoch 9), and 100.00% (epoch 4). Aggregate test per-digit/full-image accuracies were 73.72%/36.20%, 73.00%/31.80%, and 75.20%/38.80%, respectively.

**Conclusion.** The 100k run gave the strongest aggregate test result, but the gain over 10k was modest and the 20k run did not improve monotonically. More training data helped somewhat in this comparison; a large gap remains between synthetic validation performance and these fixed test results. Test metrics are reported only after validation-based checkpoint selection.

### 6.2 Data augmentation and overfitting

Compare matched runs with and without data augmentation.

**Results.** Without augmentation, the best validation full-image accuracy was 99.63% (epoch 10), while the augmented run reached 99.81% (epoch 9). Final train/validation full-image accuracies were 100.00%/99.63% without augmentation and 99.84%/99.81% with augmentation. Aggregate test per-digit/full-image accuracy improved from 52.48%/6.00% to 68.56%/24.60% with augmentation.

**Conclusion.** Augmentation slightly narrowed the final train-validation gap and substantially improved the fixed-test comparison in this run. It improved generalization, although test full-image accuracy remained well below validation accuracy. The comparison does not use test results to select a training configuration.

### 6.3 Regularization and overfitting

Compare weight decay 0, 1e-4, and 1e-3 while keeping the model and other settings fixed.

**Results.** Best validation full-image accuracy was 99.80% (wd=0), 99.87% (wd=1e-4), and 99.79% (wd=1e-3). Aggregate test per-digit/full-image accuracy was 72.44%/31.60%, 70.88%/28.00%, and 71.76%/30.80%, respectively. Final train-validation gaps were small for all three runs.

**Conclusion.** Validation performance was similar across these regularization strengths, with 1e-4 slightly highest. The fixed-test results varied without a monotonic relationship to weight decay, so these runs do not show a strong regularization trend. Checkpoints remain selected using validation full-image accuracy.

### 6.4 Batch size

Compare batch sizes 32, 64, and 128 with the same model, data, optimizer, and schedule.

**Results.** Best validation full-image accuracy was 99.91% (batch 32), 99.84% (batch 64), and 99.43% (batch 128). Aggregate test per-digit/full-image accuracy was 68.68%/24.40%, 74.44%/35.20%, and 69.76%/27.40%. End-to-end train-script wall time, measured on the same GPU and including validation/checkpoint overhead, was 3.61, 3.72, and 3.84 minutes.

**Conclusion.** Batch 32 had the highest validation accuracy and shortest measured runtime; batch 128 was slowest and had the lowest validation accuracy. Batch 64 was strongest on the fixed test comparison, which is reported for analysis and does not change validation-based selection.

### 6.5 Learning rate

Compare initial learning rates 1e-4, 1e-3, and 1e-2 under the same cosine schedule.

**Results.** Best validation full-image accuracy was 15.63% (1e-4), 99.76% (1e-3), and 0.01% (1e-2). Aggregate test per-digit/full-image accuracy was 32.40%/0.60%, 70.60%/27.40%, and 10.60%/0.00%.

**Conclusion.** The 1e-4 rate learned too slowly within 10 epochs, while 1e-2 failed to converge and remained near chance. The 1e-3 run converged reliably and gave the strongest validation and fixed-test performance of these three settings. The learning-rate choice is based on validation behavior.

### 6.6 Backbone comparison

Compare ResNet-18 and ResNet-34 under comparable settings.

**Results.** Best validation full-image accuracy was 99.92% for ResNet-18 (epoch 8) and 99.95% for ResNet-34 (epoch 7). Aggregate test per-digit/full-image accuracy was 80.32%/48.20% and 75.08%/35.60%; measured train-script time was 3.84 and 4.71 minutes. The model-cost cell reports parameter counts and approximate FLOPs at 80x260 input size.

**Conclusion.** ResNet-34 reached a marginally higher validation peak at higher training cost, while ResNet-18 performed better on the fixed test comparison. This illustrates that model capacity and validation accuracy alone do not guarantee stronger performance under the observed test distribution.

### 6.7 Global average pooling

Compare global average pooling with flattening the final spatial feature map.

**Results.** Best validation full-image accuracy was 99.97% with global average pooling and 99.95% with spatial flattening. Aggregate test per-digit/full-image accuracy was 80.16%/46.80% for global average pooling and 74.24%/34.00% for spatial flattening. Parameter counts and approximate FLOPs are reported in the model-cost table.

**Conclusion.** Preserving the spatial feature map did not improve the validation peak and performed worse on the fixed test comparison in these runs. The larger position-specific head may add capacity without enough benefit here. Test metrics are descriptive only and were not used to pick the final model.

### 6.8 Prediction visualization

The prediction grid in Section 5 shows test images with ground truth and predicted labels; titles turn red for any full-image failure, and every failure is plotted separately.

**Final test results and failure analysis.** The checkpoint was selected from the reference validation set only (epoch 10, 99.90% validation full-image accuracy).

| Test folder | Images | Per-digit accuracy | Full-image accuracy | Full-image failures |
|---|---:|---:|---:|---:|
| test0 | 100 | 100.00% | 100.00% | 0 |
| test1 | 100 | 100.00% | 100.00% | 0 |
| test2 | 100 | 100.00% | 100.00% | 0 |
| test3 | 100 | 100.00% | 100.00% | 0 |
| test4 | 100 | 100.00% | 100.00% | 0 |
| **Aggregate** | **500** | **100.00%** | **100.00%** | **0** |

There are no failure cases to analyze. The previous shifted-generator model reached 97.68% per digit and 89.00% full image, with most errors in test0 and test3. The main change since then is the font: the earlier generators used Arial-like system fonts, while the provided CAPTCHA examples use glyphs closest to Pillow's default font. The side margins, spacing, and JPEG variation were also included in the synthetic rendering ranges. The validation set has 10 full-image errors in 10,000 images (99.90%). The generator was designed by referring to the assignment's provided examples, so the 100% score describes these required groups and is not an independent estimate for a new CAPTCHA distribution. Test labels and accuracy did not influence training or checkpoint selection. The aggregate assignment score from the provided formula is 40.00.

In [ ]:
# Save the evaluation computed above next to the selected checkpoint.
(run_dir / "final_test_results.json").write_text(json.dumps(final_test_report, indent=2) + "\n", encoding="utf-8")
print(f"Saved evaluation report: {run_dir / 'final_test_results.json'}")